## Thực hành 1: Khai thác dataset thực tế

In [1]:
#import thư viện
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

### Tổng quan về Bộ dữ liệu AI4I 2020 Predictive Maintenance
- Bộ dữ liệu AI4I 2020 Predictive Maintenance Dataset (Bảo trì dự đoán AI4I 2020) được thiết kế bởi tác giả Stephan Matzka và công bố tại Viện Nghiên cứu Trí tuệ Nhân tạo về Công nghiệp (AI4I). Đây là một tập dữ liệu chuẩn hóa (synthetic dataset) mô phỏng lại các kịch bản vận hành thực tế của máy móc thiết bị trong một nhà máy sản xuất công nghiệp.
Mục tiêu cốt lõi của bộ dữ liệu này là giúp các kỹ sư và nhà khoa học dữ liệu xây dựng, thử nghiệm các mô hình học máy (Machine Learning) nhằm dự đoán trước thời điểm thiết bị sẽ bị hỏng hóc (Predictive Maintenance) dựa trên các tín hiệu cảm biến.
------------------------------
### Cấu trúc và Ý nghĩa các Cột Thuộc tính
Bộ dữ liệu gồm có 10.000 dòng (mẫu thử) và 14 cột dữ liệu, được chia thành các nhóm thông tin chính sau đây:
### 1. Nhóm thông tin định danh và Phân loại

* UDI: Số thứ tự định danh duy nhất cho từng bản ghi dữ liệu (từ 1 đến 10.000).
* Product ID: Mã định danh của sản phẩm, bắt đầu bằng ký tự đại diện cho loại máy.
* Type: Phân loại chất lượng của dòng máy vận hành, gồm 3 cấp độ:
* L (Low): Dòng máy phổ thông, chiếm phần lớn dữ liệu (60%), có tỷ lệ hỏng hóc cao hơn.
   * M (Medium): Dòng máy trung cấp (chiếm 30%).
   * H (High): Dòng máy cao cấp (chiếm 10%), được chế tạo với tiêu chuẩn khắt khe, ít hỏng hóc hơn.

### 2. Nhóm thông số kỹ thuật (Dữ liệu từ cảm biến)

* Air temperature [K]: Nhiệt độ không khí/môi trường xung quanh nhà máy tính theo độ Kelvin (°K).
* Process temperature [K]: Nhiệt độ của chính quy trình sản xuất bên trong máy tính theo độ Kelvin (°K).
* Rotational speed [rpm]: Tốc độ quay của trục máy, tính bằng số vòng trên phút (rpm).
* Torque [Nm]: Lực xoắn tác động lên trục máy, tính bằng đơn vị Newton-mét (Nm).
* Tool wear [min]: Thời gian mài mòn của công cụ cắt/gọt trong quá trình gia công, tính theo số phút.

### 3. Cột mục tiêu (Target) để dự đoán

* Machine failure: Trạng thái hỏng máy. Cột này nhận giá trị 0 nếu máy chạy bình thường và 1 nếu xảy ra sự cố hỏng hóc tại thời điểm đó.

### 4. Nhóm phân loại chi tiết nguyên nhân sự cố
Nếu cột Machine failure bằng 1, dữ liệu sẽ chỉ rõ cụ thể thiết bị hỏng vì lỗi gì trong 5 nhãn lỗi nhị phân sau đây:

* TWF (Tool Wear Failure): Lỗi do công cụ bị mài mòn quá mức quy định.
* HDF (Heat Dissipation Failure): Lỗi do tản nhiệt kém (xảy ra khi chênh lệch giữa nhiệt độ môi trường và nhiệt độ quy trình quá thấp, khiến máy bị quá nhiệt).
* PWF (Power Failure): Lỗi do công suất vận hành (tích của mô-men xoắn và tốc độ quay) nằm ngoài khoảng an toàn.
* OSF (Overstrain Failure): Lỗi do máy bị quá tải (sự kết hợp giữa lực xoắn cao và thời gian mài mòn dao lớn).
* RNF (Random Failure): Lỗi xảy ra ngẫu nhiên, không thể giải thích bằng các thông số cảm biến thông thường.

------------------------------
### Tại sao bộ dữ liệu này lại quan trọng và hay được sử dụng?

* Tính mất cân bằng dữ liệu (Imbalanced Data): Trong thực tế, máy móc chạy bình thường chiếm đa số, sự cố hỏng hóc xảy ra rất ít. Trong 10.000 dòng của tập dữ liệu này, chỉ có khoảng 339 ca hỏng máy (tỷ lệ ~3.39%). Bài toán này ép người học phải xử lý kỹ thuật mất cân bằng nhãn trước khi huấn luyện mô hình.
* Bài toán Đa nhãn (Multi-label Classification): Một chiếc máy có thể hỏng do 1 lỗi, hoặc đồng thời bị hỏng bởi 2 lỗi khác nhau (ví dụ: vừa bị quá nhiệt HDF, vừa bị mài mòn dao TWF). Điều này giúp sinh viên tiếp cận bài toán phân loại đa nhãn phức tạp hơn.
* Mối quan hệ phi tuyến giữa các biến: Tốc độ quay và lực xoắn thường tỷ lệ nghịch với nhau. Nhiệt độ môi trường ảnh hưởng trực tiếp đến nhiệt độ hệ thống. Các đặc trưng này rất thích hợp để thực hành kỹ thuật tạo tính năng mới (Feature Engineering) trên NumPy và Pandas.

In [2]:
import os
import shutil
import kagglehub
import pandas as pd

# 1. file_path lưu dataset
target_dir = r"C:\Users\Admin\Desktop\Project\DataAnalytics\Lab1"
os.makedirs(target_dir, exist_ok=True) # Tự động tạo thư mục nếu chưa có

downloaded_path = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")

# 3. Di chuyển dataset vừa tải về thư mục đích (target_dir)
filename = "predictive_maintenance_dataset_ai4i_2020.csv"
src_file = os.path.join(downloaded_path, filename)
dest_file = os.path.join(target_dir, filename)

# Nếu file chưa tồn tại ở thư mục đích, tiến hành sao chép sang
if os.path.exists(src_file):
    shutil.copy(src_file, dest_file)
    print(f"Đã tải và lưu file thành công tại: {dest_file}")
else:
    # Trường hợp Kaggle đổi tên file bên trong, ta sẽ quét tìm file .csv bất kỳ để di chuyển
    for file in os.listdir(downloaded_path):
        if file.endswith('.csv'):
            shutil.copy(os.path.join(downloaded_path, file), os.path.join(target_dir, file))
            dest_file = os.path.join(target_dir, file)
            print(f"Đã tải và lưu file thành công tại: {dest_file}")
            break

# 4. Đọc dữ liệu lên để làm bài tập thực hành
df = pd.read_csv(dest_file)
print("\n--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
display(df.head())

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 136k/136k [00:00<00:00, 303kB/s]

Extracting files...
Đã tải và lưu file thành công tại: C:\Users\Admin\Desktop\Project\DataAnalytics\Lab1\ai4i2020.csv

--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### Tổng quan về Bộ dữ liệu Steel Plates Faults
Bộ dữ liệu Steel Plates Faults (được đăng tải bởi tác giả noepinefrin trên Kaggle) có nguồn gốc từ Kho lưu trữ Học máy UCI danh tiếng, do Viện Nghiên cứu Semeion (Ý) đóng góp. Khác với bộ AI4I 2020 tập trung vào tín hiệu cảm biến thời gian thực của máy móc, bộ dữ liệu này đại diện cho một bài toán kiểm định chất lượng sản phẩm đầu ra (Quality Control) trong ngành công nghiệp nặng, cụ thể là sản xuất thép không gỉ cán nguội.
Mục tiêu cốt lõi của bộ dữ liệu này là giúp các kỹ sư xây dựng hệ thống thị giác máy tính và học máy (Computer Vision & Machine Learning) để tự động nhận dạng, phân loại các lỗi khuyết tật trên bề mặt tấm thép dựa trên các đặc trưng hình học và độ sáng.

### Cấu trúc và Ý nghĩa các Cột Thuộc tính
Bộ dữ liệu bao gồm 1.941 dòng (mẫu thử) và 34 cột, được chia làm hai nhóm thông tin cực kỳ rõ rệt: 27 đặc trưng hình ảnh (Features) và 7 cột nhãn lỗi mục tiêu (Targets).
### 1. Nhóm 27 đặc trưng hình học và quang học (Dữ liệu đầu vào)
Khi tấm thép chạy trên băng tải, hệ thống camera công nghiệp sẽ chụp ảnh và thuật toán xử lý ảnh sẽ tự động trích xuất ra các thông số số học sau:

* Tọa độ vị trí lỗi: X_Minimum, X_Maximum, Y_Minimum, Y_Maximum (Xác định khung biên tọa độ pixel của vết khuyết tật trên bề mặt).
* Kích thước hình học: Pixels_Areas (Diện tích vùng bị lỗi tính bằng pixel), X_Perimeter (Chu vi vùng lỗi theo chiều ngang), Y_Perimeter (Chu vi vùng lỗi theo chiều dọc).
* Độ sáng & Độ tương phản: Sum_of_Luminosity (Tổng lượng ánh sáng phản xạ), Minimum_of_Luminosity, Maximum_of_Luminosity, Luminosity_Index (Chỉ số độ sáng tổng quan nhằm phân biệt vết xước sâu hay chỉ là vết ố bề mặt).
* Các chỉ số hình dáng nâng cao (Shape Indexes): Length_of_Conveyer (Chiều dài băng tải), Steel_Plate_Thickness (Độ dày tấm thép), Edges_Index, Empty_Index, Square_Index, Outside_X_Index, Edges_X_Index, Edges_Y_Index (Các tỷ lệ toán học giúp máy tính hiểu vết lỗi có dạng hình tròn, hình vuông hay vệt dài).
* Các thuộc tính Logarit: Log_of_Areas, Log_X_Index, Log_Y_Index (Giá trị đã được biến đổi logarit để làm giảm độ lệch của dữ liệu kích thước).
* Mác thép: TypeOfSteel_A300, TypeOfSteel_A400 (Nhãn nhị phân 0/1 thể hiện tấm thép thuộc loại mác thép nào).

### 2. Nhóm 7 nhãn lỗi bề mặt mục tiêu (Dữ liệu đầu ra - Target)
Hệ thống mã hóa dữ liệu dưới dạng One-Hot Encoding phân chia thành 7 cột cuối. Nếu dòng dữ liệu thuộc loại lỗi nào thì cột đó nhận giá trị 1, các cột còn lại nhận giá trị 0:

   1. Pastry: Lỗi nếp gấp/thắt nút bề mặt (do quá trình cán hoặc dị vật đè lên phôi).
   2. Z_Scratch: Vết xước dọc theo hướng di chuyển của băng tải (trục Z).
   3. K_Scratch: Vết xước chéo, xước sâu (K-type scratch).
   4. Stains: Vết ố bẩn, rỉ sét do dính dầu mỡ hoặc hóa chất bảo quản.
   5. Dirtiness: Vết bẩn do tạp chất hạt thô bám vào trong quá trình nguội.
   6. Bumps: Vết lồi lõm, gồ ghề cục bộ do trục cán bị mẻ hoặc bọt khí ngậm trong thép.
   7. Other_Faults: Nhóm các loại lỗi bề mặt phức tạp khác chưa thể phân định vào 6 nhóm trên.


### Tại sao bộ dữ liệu này lại quan trọng và hay được sử dụng?

* Không có dữ liệu khuyết thiếu (No Missing Values): Toàn bộ 1.941 dòng dữ liệu đều hoàn chỉnh, không có giá trị rỗng (NaN). Điều này giúp người mới học tập trung 100% vào việc thực hành các hàm tính toán của NumPy mà không bị phân tâm bởi việc điền khuyết dữ liệu.
* Bài toán Phân loại đa lớp từ mã hóa One-Hot: Bộ dữ liệu kiểm tra tư duy logic toán học của học viên. Họ phải dùng NumPy để xử lý cấu trúc đa nhãn nhị phân này, hoặc dùng các hàm tìm vị trí cực đại để nén 7 cột mục tiêu thành 1 cột nhãn duy nhất (chứa các số từ 0 đến 6).
* Thách thức xử lý Ngoại lai (Outliers): Vì là dữ liệu đo đạc hình học từ ảnh, các biến như Pixels_Areas (Diện tích) hay X_Perimeter có độ lệch phân phối rất lớn (có những vết lỗi siêu nhỏ nhưng có những vết xước kéo dài toàn tấm thép). Đây là môi trường hoàn hảo để sinh viên thực hành kỹ thuật lọc phân vị bằng np.percentile() và cắt biên dữ liệu bằng np.clip().

In [8]:
# 1. file_path lưu dataset
target_dir = r"C:\Users\Admin\Desktop\Project\DataAnalytics\Lab1"
os.makedirs(target_dir, exist_ok=True) # Tự động tạo thư mục nếu chưa có

downloaded_path = kagglehub.dataset_download("noepinefrin/steel-plates-faults-dataset")

# 3. Di chuyển dataset vừa tải về thư mục đích (target_dir)
filename = "steel_plates_faults_dataset.csv"
src_file = os.path.join(downloaded_path, filename)
dest_file = os.path.join(target_dir, filename)

# Nếu file chưa tồn tại ở thư mục đích, tiến hành sao chép sang
if os.path.exists(src_file):
    shutil.copy(src_file, dest_file)
    print(f"Đã tải và lưu file thành công tại: {dest_file}")
else:
    # Trường hợp Kaggle đổi tên file bên trong, ta sẽ quét tìm file .csv bất kỳ để di chuyển
    for file in os.listdir(downloaded_path):
        if file.endswith('.csv'):
            shutil.copy(os.path.join(downloaded_path, file), os.path.join(target_dir, file))
            dest_file = os.path.join(target_dir, file)
            print(f"Đã tải và lưu file thành công tại: {dest_file}")
            break

# 4. Đọc dữ liệu lên để làm bài tập thực hành
df = pd.read_csv(dest_file)
print("\n--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
display(df.head())

Đã tải và lưu file thành công tại: C:\Users\Admin\Desktop\Project\DataAnalytics\Lab1\steel_plates_faults_original_dataset.csv

--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---


,id,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,0,42,50,270900,270944,267,17,44,24220,76,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,1,645,651,2538079,2538108,108,10,30,11397,84,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,2,829,835,1553913,1553931,71,8,19,7972,99,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,3,853,860,369370,369415,176,13,45,18996,99,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,4,1289,1306,498078,498335,2409,60,260,246930,37,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


In [11]:
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

target_dir = r"C:\Users\Admin\Desktop\Project\DataAnalytics\Lab1"
os.makedirs(target_dir, exist_ok=True)
file_name = "data_gov_air_quality.csv"
dest_file = os.path.join(target_dir, file_name)

# Đường dẫn dữ liệu mở CSV từ cổng Data.gov
url = "https://technologypublisher.com"

try:
    # GIẢI PHÁP SỬA LỖI 403: Tạo Header giả lập trình duyệt Chrome để hack tường lửa máy chủ
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    req = urllib.request.Request(url, headers=headers)
    
    # Tiến hành tải dữ liệu qua luồng trình duyệt giả lập
    with urllib.request.urlopen(req) as response:
        # Đọc dữ liệu trực tiếp vào Pandas
        df = pd.read_csv(response)
        
    # Lưu bản sao file CSV cục bộ vào thư mục ổ D của bạn để lưu trữ bản sao
    df.to_csv(dest_file, index=False)
    print(f"Tải và cấu hình lưu file thành công tại: {dest_file}")

except Exception as e:
    print(f"\nGặp lỗi tải trực tuyến: {e}")
    print("Đang kiểm tra file cứu cánh có sẵn tại ổ D...")
    if os.path.exists(dest_file):
        df = pd.read_csv(dest_file)
        print("Đã đọc file có sẵn từ ổ D thành công!")
    else:
        raise Exception("Không thể lấy dữ liệu trực tuyến và file cục bộ không tồn tại.")


Gặp lỗi tải trực tuyến: HTTP Error 404: Not Found
Đang kiểm tra file cứu cánh có sẵn tại ổ D...


Exception: Không thể lấy dữ liệu trực tuyến và file cục bộ không tồn tại.